# Uji Awal Pemilihan Algoritma — Kelompok 3
**Mata Kuliah:** Machine Learning (Kelas C) — Bapak Adi Purnawan

**Judul:** Prediksi Risiko Stroke Menggunakan Logistic Regression dan Gradient Boosting dengan Interpretasi Explainable AI

**Anggota:** Deliana Br Manalu (2305551036) · Ravi Arnan Irianto (2305551076) · Ezza Putra Wibawa (2305551144) · Devin (2305551173)

---

Notebook ini menjawab satu pertanyaan: **algoritma mana yang pantas dicantumkan di judul?**

Kami menguji enam algoritma terhadap tiga strategi penanganan ketidakseimbangan kelas,
lalu menguji satu hipotesis tambahan tentang ambang keputusan.

Hasilnya masih sementara — hyperparameter belum disetel dan preprocessing belum final.
Yang dicari di sini hanyalah urutan peringkat, bukan angka akhir.


## 1. Persiapan

In [1]:
# imbalanced-learn belum tersedia bawaan di Colab.
# Dipasang hanya kalau memang belum ada, supaya sel ini aman
# dijalankan di lingkungan mana pun.
try:
    import imblearn
    print("imbalanced-learn sudah tersedia:", imblearn.__version__)
except ImportError:
    !pip install -q imbalanced-learn

imbalanced-learn sudah tersedia: 0.14.2


In [2]:
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

SEED = 42
METRIK = ["recall", "precision", "f1", "roc_auc"]
URL = ("https://raw.githubusercontent.com/ray-project/raydp/master/"
       "tutorials/dataset/healthcare-dataset-stroke-data.csv")

In [3]:
d = pd.read_csv(URL).drop(columns=["id"])
d["bmi"] = d.bmi.fillna(d.bmi.median())        # imputasi sementara, dibahas serius di notebook preprocessing

X = pd.get_dummies(d.drop(columns=["stroke"]), drop_first=True)
y = d.stroke

print(f"Data  : {X.shape[0]} baris, {X.shape[1]} fitur setelah encoding")
print(f"Positif (stroke): {y.sum()} ({y.mean()*100:.2f}%)")

Data  : 5110 baris, 16 fitur setelah encoding
Positif (stroke): 249 (4.87%)


## 2. Enam Algoritma × Tiga Strategi

SMOTE diletakkan **di dalam** pipeline supaya hanya diterapkan pada lipatan latih.
Kalau diterapkan di luar, data uji ikut disintesis dan hasilnya jadi bohong — ini
kesalahan yang sering terjadi di penelitian sejenis.

KNN dan Gradient Boosting tidak menyediakan parameter `class_weight`, jadi kombinasi
tersebut dilewati.

In [4]:
def model_dasar(seed=SEED):
    """(nama, konstruktor, dukung_class_weight)"""
    return [
        ("Logistic Regression", lambda **k: LogisticRegression(max_iter=2000, random_state=seed, **k), True),
        ("Gradient Boosting",   lambda **k: GradientBoostingClassifier(random_state=seed, **k), False),
        ("KNN",                 lambda **k: KNeighborsClassifier(n_neighbors=5, **k), False),
        ("Decision Tree",       lambda **k: DecisionTreeClassifier(random_state=seed, **k), True),
        ("Random Forest",       lambda **k: RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1, **k), True),
        ("SVM (RBF)",           lambda **k: SVC(random_state=seed, **k), True),
    ]


def bangun(konstruktor, strategi, dukung_cw):
    """Rangkai pipeline sesuai strategi. None kalau kombinasi tidak berlaku."""
    langkah = [("skala", StandardScaler())]
    if strategi == "SMOTE":
        langkah.append(("smote", SMOTE(random_state=SEED)))
        clf = konstruktor()
    elif strategi == "class_weight":
        if not dukung_cw:
            return None
        clf = konstruktor(class_weight="balanced")
    else:
        clf = konstruktor()
    langkah.append(("model", clf))
    return Pipeline(langkah)

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
baris = []

for nama, konstruktor, dukung_cw in model_dasar():
    for strategi in ["tanpa penanganan", "class_weight", "SMOTE"]:
        pipe = bangun(konstruktor, strategi, dukung_cw)
        if pipe is None:
            print(f"{nama:20} | {strategi:16} | tidak mendukung, dilewati")
            continue
        hasil = cross_validate(pipe, X, y, cv=cv, scoring=METRIK, n_jobs=-1)
        b = {"Algoritma": nama, "Strategi": strategi}
        b.update({m: hasil[f"test_{m}"].mean() for m in METRIK})
        baris.append(b)
        print(f"{nama:20} | {strategi:16} | recall {b['recall']:.3f}  "
              f"precision {b['precision']:.3f}  F1 {b['f1']:.3f}  AUC {b['roc_auc']:.3f}")

tabel = pd.DataFrame(baris).round(3)

Logistic Regression  | tanpa penanganan | recall 0.004  precision 0.200  F1 0.008  AUC 0.837


Logistic Regression  | class_weight     | recall 0.795  precision 0.134  F1 0.229  AUC 0.837


Logistic Regression  | SMOTE            | recall 0.787  precision 0.137  F1 0.233  AUC 0.835


Gradient Boosting    | tanpa penanganan | recall 0.008  precision 0.120  F1 0.014  AUC 0.839
Gradient Boosting    | class_weight     | tidak mendukung, dilewati


Gradient Boosting    | SMOTE            | recall 0.414  precision 0.145  F1 0.215  AUC 0.798
KNN                  | tanpa penanganan | recall 0.036  precision 0.299  F1 0.064  AUC 0.598
KNN                  | class_weight     | tidak mendukung, dilewati


...

Random Forest        | class_weight     | recall 0.064  precision 0.124  F1 0.085  AUC 0.814


Random Forest        | SMOTE            | recall 0.133  precision 0.117  F1 0.124  AUC 0.788


SVM (RBF)            | tanpa penanganan | recall 0.000  precision 0.000  F1 0.000  AUC 0.625


SVM (RBF)            | class_weight     | recall 0.578  precision 0.115  F1 0.192  AUC 0.771


SVM (RBF)            | SMOTE            | recall 0.482  precision 0.120  F1 0.192  AUC 0.753


In [6]:
tabel.sort_values("recall", ascending=False).head(8)

,Algoritma,Strategi,recall,precision,f1,roc_auc
1,Logistic Regression,class_weight,0.795,0.134,0.229,0.837
2,Logistic Regression,SMOTE,0.787,0.137,0.233,0.835
14,SVM (RBF),class_weight,0.578,0.115,0.192,0.771
15,SVM (RBF),SMOTE,0.482,0.120,0.192,0.753
4,Gradient Boosting,SMOTE,0.414,0.145,0.215,0.798
6,KNN,SMOTE,0.305,0.089,0.137,0.620
9,Decision Tree,SMOTE,0.237,0.107,0.147,0.568
7,Decision Tree,tanpa penanganan,0.185,0.160,0.171,0.567


### Pengamatan

Dua hal yang langsung terlihat:

1. **Logistic Regression menang**, mengalahkan Random Forest, SVM, dan Gradient Boosting.
   Model paling sederhana justru paling baik.
2. **AUC bisa menyesatkan juga.** Gradient Boosting tanpa penanganan punya AUC tertinggi
   tetapi recall-nya hampir nol — ia memberi peringkat yang bagus, tetapi ambang
   bawaannya 0,50 tidak pernah terlampaui oleh pasien mana pun.

In [7]:
tabel.sort_values("roc_auc", ascending=False).head(6)

,Algoritma,Strategi,recall,precision,f1,roc_auc
3,Gradient Boosting,tanpa penanganan,0.008,0.120,0.014,0.839
0,Logistic Regression,tanpa penanganan,0.004,0.200,0.008,0.837
1,Logistic Regression,class_weight,0.795,0.134,0.229,0.837
2,Logistic Regression,SMOTE,0.787,0.137,0.233,0.835
11,Random Forest,class_weight,0.064,0.124,0.085,0.814
10,Random Forest,tanpa penanganan,0.008,0.300,0.016,0.802


## 3. Hipotesis: Apakah SMOTE Benar-benar Diperlukan?

Perhatikan bahwa AUC Logistic Regression **tidak berubah** oleh strategi apa pun.
AUC mengukur kemampuan model **mengurutkan** pasien dari paling berisiko ke paling aman.
Kalau urutannya tidak berubah, berarti `class_weight` dan SMOTE tidak membuat model
lebih pintar — keduanya hanya menggeser **ambang keputusan**.

Kalau dugaan ini benar, cukup geser ambangnya langsung, tanpa perlu membangkitkan
ribuan baris data sintetis. Mari dibuktikan.

In [8]:
def probabilitas_lipatan(buat_model, X, y, cv):
    """Kumpulkan probabilitas out-of-fold supaya setiap baris diprediksi oleh model
    yang tidak pernah melihatnya."""
    prob = np.zeros(len(y))
    for latih, uji in cv.split(X, y):
        m = buat_model().fit(X.iloc[latih], y.iloc[latih])
        prob[uji] = m.predict_proba(X.iloc[uji])[:, 1]
    return prob


def ambang_untuk_recall(y, prob, target=0.80):
    """Ambang tertinggi yang masih mencapai recall >= target."""
    for t in sorted(np.unique(np.round(prob, 4)), reverse=True):
        if recall_score(y, (prob >= t).astype(int)) >= target:
            return t
    return 0.5

In [9]:
MODEL_UJI = {
    "Logistic Regression": lambda: Pipeline([("skala", StandardScaler()),
                                             ("model", LogisticRegression(max_iter=2000, random_state=SEED))]),
    "Gradient Boosting":   lambda: Pipeline([("skala", StandardScaler()),
                                             ("model", GradientBoostingClassifier(random_state=SEED))]),
    "Random Forest":       lambda: Pipeline([("skala", StandardScaler()),
                                             ("model", RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))]),
}

hasil_ambang = []
for nama, buat in MODEL_UJI.items():
    prob = probabilitas_lipatan(buat, X, y, cv)
    t = ambang_untuk_recall(y, prob)
    for label, ambang in [("default 0,50", 0.5), (f"disetel {t:.3f}", t)]:
        pred = (prob >= ambang).astype(int)
        hasil_ambang.append({
            "Model": nama, "Ambang": label,
            "recall": recall_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred),
            "roc_auc": roc_auc_score(y, prob),
        })

pd.DataFrame(hasil_ambang).round(3)

,Model,Ambang,recall,precision,f1,roc_auc
0,Logistic Regression,"default 0,50",0.004,0.500,0.008,0.837
1,Logistic Regression,disetel 0.048,0.803,0.134,0.229,0.837
2,Gradient Boosting,"default 0,50",0.008,0.111,0.015,0.837
3,Gradient Boosting,disetel 0.043,0.803,0.135,0.231,0.837
4,Random Forest,"default 0,50",0.008,0.222,0.016,0.802
5,Random Forest,disetel 0.035,0.815,0.110,0.194,0.802


### Hipotesis terbukti

Menyetel ambang keputusan menghasilkan angka yang **praktis identik dengan SMOTE**,
tanpa membangkitkan satu pun baris data palsu. SMOTE menambah sekitar 4.600 baris
sintetis untuk hasil yang sama.

Perhatikan juga kolom `roc_auc`: nilainya sama persis untuk ambang default maupun
ambang disetel. Itu menegaskan bahwa ambang **tidak mengubah kemampuan model**, hanya
mengubah titik potongnya.

Temuan ini menjadi salah satu kontribusi utama laporan kami.

## 4. Kesimpulan

| Algoritma | Strategi terbaik | Recall | F1 | AUC |
|---|---|---|---|---|
| **Gradient Boosting** | penyetelan ambang | 0,803 | **0,231** | **0,837** |
| **Logistic Regression** | penyetelan ambang | 0,803 | 0,229 | **0,837** |
| SVM (RBF) | `class_weight` | 0,578 | 0,192 | 0,771 |
| KNN | SMOTE | 0,305 | 0,137 | 0,620 |
| Decision Tree | SMOTE | 0,237 | 0,147 | 0,568 |
| Random Forest | SMOTE | 0,133 | 0,124 | 0,788 |

**Dua algoritma terpilih untuk judul: Logistic Regression dan Gradient Boosting.**
Sengaja dipilih dua yang berbeda jenis — satu linear, satu ensemble boosting — supaya
ada kontras yang bisa dibahas.

Random Forest, yang paling sering dipuji di literatur, justru melewatkan 87% pasien
stroke pada dataset ini. Keempat algoritma lain tetap dilaporkan sebagai pembanding
di bab hasil.

**Strategi ketidakseimbangan bertambah jadi empat:** tanpa penanganan, `class_weight`,
SMOTE, dan penyetelan ambang keputusan.